# Stage 1 — Informed minority replication (Oct–Dec 2024)

Orchestration and prerequisites: **`docs/SETUP.md`**.

Pipeline: Gamma → R2 metadata → Goldsky `order_filled` → R2 → DuckDB analysis (PnL, event-level sign randomization, classification, OIB regression).

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from dotenv import load_dotenv

ROOT = Path.cwd()
if (ROOT / "pyproject.toml").exists():
    pass
elif (ROOT.parent / "pyproject.toml").exists():
    os.chdir(ROOT.parent)
load_dotenv()

from src.lib.duck import get_duck, r2_uri

print("CWD", Path.cwd())
print("R2_BUCKET", os.environ.get("R2_BUCKET", "(unset)"))

In [ ]:
def run_stage1_modules(run: bool = False):
    """Set run=True to execute pipeline steps (requires R2 + network)."""
    if not run:
        print("Skipping module execution (run=False). Run from CLI per docs/SETUP.md.")
        return
    from src.data import load_markets
    from src.data import render_pipeline_yaml
    from src.data import verify_ingest
    from src.analysis import pnl, sign_randomization, classification, oib_regression

    load_markets.main()
    render_pipeline_yaml.main()
    print("Deploy: goldsky turbo apply goldsky/order_filled_backfill.rendered.yaml")
    verify_ingest.main()
    pnl.main()
    sign_randomization.main()
    classification.main()
    oib_regression.main()


run_stage1_modules(run=False)

In [ ]:
# PnL histogram by trader group (requires labels on R2)
con = get_duck()
try:
    acct = con.execute(f"SELECT * FROM read_parquet('{r2_uri('derived', 'pnl_account.parquet')}')").df()
    lbl = con.execute(f"SELECT * FROM read_parquet('{r2_uri('labels', 'account_classification.parquet')}')").df()
    m = acct.merge(lbl[["user_id", "trader_group"]], on="user_id", how="left")
    fig, ax = plt.subplots(figsize=(8, 4))
    for g in m["trader_group"].dropna().unique():
        sub = m.loc[m["trader_group"] == g, "total_pnl"].clip(-5_000, 5_000)
        if len(sub) > 0:
            ax.hist(sub, bins=40, alpha=0.35, label=str(g), density=True)
    ax.set_xlabel("total_pnl (clipped ±5000)")
    ax.legend(fontsize=7)
    ax.set_title("Realized PnL by trader_group")
    plt.show()
except Exception as e:
    print("Skip plot:", e)

In [ ]:
# Skilled share banner + go / no-go (reads classification parquet)
try:
    con = get_duck()
    lbl = con.execute(f"SELECT * FROM read_parquet('{r2_uri('labels', 'account_classification.parquet')}')").df()
    classified = lbl[lbl["trader_group"] != "Excluded"]
    share = (lbl["trader_group"] == "Skilled_Winner").mean()
    share_c = (lbl["trader_group"] == "Skilled_Winner").sum() / max(len(classified), 1)
    print(f"Skilled share (all rows): {share:.2%}")
    print(f"Skilled share / non-excluded: {share_c:.2%}")
    band_ok = 0.02 <= share_c <= 0.06
    print("BAND_OK (2–6% on non-excluded):", band_ok)
except Exception as e:
    print("Classification not available:", e)

In [ ]:
# Run Report: concatenate recent JSONL debug logs
from glob import glob

lines = []
for path in sorted(glob("logs/*.jsonl"))[-15:]:
    lines.append(f"## {path}\n")
    with open(path, encoding="utf-8") as f:
        tail = f.readlines()[-40:]
    lines.extend(tail)
    lines.append("\n")
report = "".join(lines)
print(report[:8000] if len(report) > 8000 else report)